# First experiments with whole Project architecture setup

In [1]:
%load_ext autoreload
%autoreload 2
%load_ext tensorboard

In [2]:
from constraints.lightning_wrappers.modules import ProjectLightning
from constraints.datatools.datasets import CachedArtificalDataset
from constraints import get_experiment_folder, get_data_folder, show_torch_image
from constraints.transforms.transformers import RigidTransformer
from constraints.computers.loss_computers import ProjectLossComputer
from constraints.losses import OneSideSDFSquare
from constraints.models.affine import ProjectWithTemplateA 
from constraints.computers.loss_computers import CrossEntrAndOneSide

from pathlib import Path

import torch
import pytorch_lightning as pl
FODLER = get_experiment_folder(Path("ex3")/"project_debug")
DATA =get_data_folder() / "artificial" / "downloaded"
TRN_FOLDER = DATA / "trn" / "affine"
VAL_FOLDER = DATA / "val" / "affine"

/mnt/appl/software/protobuf-python/6.31.1-GCCcore-14.2.0/lib/python3.13/site-packages/google/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


In [4]:
TB_LOGDIR = FODLER / "tb"
print(f"Experiment folder: {FODLER}")
print(f"Tensorboard logdir: {TB_LOGDIR.iterdir()}")
%tensorboard --logdir $TB_LOGDIR

Experiment folder: /mnt/personal/mrkosmic/synced/constraints/outputs/notebooks/ex3/project_debug
Tensorboard logdir: <map object at 0x7fc9b594b490>


Reusing TensorBoard on port 6006 (pid 1480619), started 0:00:25 ago. (Use '!kill 1480619' to kill it.)

In [6]:
trn_dataset = CachedArtificalDataset(TRN_FOLDER, sdf_mode="scipy")
val_dataset = CachedArtificalDataset(VAL_FOLDER, sdf_mode="scipy")

In [7]:
transformer = RigidTransformer()
loss_computer = CrossEntrAndOneSide()
net = ProjectWithTemplateA(max_translation=0.5)
module = ProjectLightning(net, transformer, loss_computer)



model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [9]:
from torch.utils.data import DataLoader
from pytorch_lightning.loggers import TensorBoardLogger

BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 20
LR = 1e-3

trn_loader = DataLoader(
    trn_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

# ProjectLightning in this repo does not define configure_optimizers, so wire it here.
module.configure_optimizers = lambda: torch.optim.Adam(module.parameters(), lr=LR)

tb_logger = TensorBoardLogger(save_dir=str(FODLER), name="tb")

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    devices="auto",
    logger=tb_logger,
    log_every_n_steps=1,
    enable_checkpointing=False,
)

trainer.fit(module, train_dataloaders=trn_loader, val_dataloaders=val_loader)



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name              | Type                 | Params | Mode 
-------------------------------------------------------------------
0 | model             | ProjectWithTemplateA | 35.8 M | train
1 | spatial_transform | RigidTransformer     | 0      | train
2 | loss_computer     | CrossEntrAndOneSide  | 0      | train
-------------------------------------------------------------------
35.8 M    Trainable params
0         Non-trainable params
35.8 M    Total params
143.124   Total estimated model params size (MB)
320       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.
